# Medical Annotation API Testing

This notebook tests the FastAPI backend using httpx for HTTP requests.

## Prerequisites:
- FastAPI server must be running: `python run.py`
- Or: `uvicorn src.main:app --reload`

## Quick Guide:
1. **Connection** - Connect to the running API
2. **Create Documents** - Add medical text documents via API
3. **Create Annotations** - Set up annotation sessions
4. **Add Entities** - Tag entities through API endpoints
5. **Add Relations** - Create relationships between entities
6. **Query & Display** - Retrieve and view data


## Setup & Configuration


In [1]:
import json

import httpx
import pandas as pd

# API Configuration
API_BASE_URL = "http://localhost:8000"
API_V1_URL = f"{API_BASE_URL}/api/v1"

print(f"📡 API Base URL: {API_BASE_URL}")
print(f"📡 API V1 URL: {API_V1_URL}")
print("\n✅ Imports successful")


📡 API Base URL: http://localhost:8000
📡 API V1 URL: http://localhost:8000/api/v1

✅ Imports successful


In [2]:
def check_api_health():
    """Check if API is running and healthy."""
    try:
        with httpx.Client() as client:
            response = client.get(f"{API_BASE_URL}/health")
            if response.status_code == 200:
                print("✅ API is healthy and running!")
                data = response.json()
                print(f"   Status: {data.get('status')}")
                return True
            else:
                print(f"❌ API returned status {response.status_code}")
                return False
    except Exception as e:
        print(f"❌ Cannot connect to API: {e}")
        print(f"   Make sure the API is running at {API_BASE_URL}")
        print("   Run: python run.py")
        return False

# Check API health
is_healthy = check_api_health()


✅ API is healthy and running!
   Status: healthy


## Helper Functions for API Calls


In [3]:
def api_post(endpoint: str, data: dict) -> dict | None:
    """Make a POST request to the API."""
    try:
        with httpx.Client() as client:
            url = f"{API_V1_URL}{endpoint}"
            response = client.post(url, json=data)

            if response.status_code in [200, 201]:
                return response.json()
            else:
                print(f"❌ POST {endpoint} failed with status {response.status_code}")
                print(f"   Response: {response.text}")
                return None
    except Exception as e:
        print(f"❌ Error: {e}")
        return None


def api_get(endpoint: str, params: dict | None = None) -> dict | list | None:
    """Make a GET request to the API."""
    try:
        with httpx.Client() as client:
            url = f"{API_V1_URL}{endpoint}"
            response = client.get(url, params=params)

            if response.status_code == 200:
                return response.json()
            else:
                print(f"❌ GET {endpoint} failed with status {response.status_code}")
                print(f"   Response: {response.text}")
                return None
    except Exception as e:
        print(f"❌ Error: {e}")
        return None


def api_put(endpoint: str, data: dict) -> dict | None:
    """Make a PUT request to the API."""
    try:
        with httpx.Client() as client:
            url = f"{API_V1_URL}{endpoint}"
            response = client.put(url, json=data)

            if response.status_code == 200:
                return response.json()
            else:
                print(f"❌ PUT {endpoint} failed with status {response.status_code}")
                print(f"   Response: {response.text}")
                return None
    except Exception as e:
        print(f"❌ Error: {e}")
        return None


def api_delete(endpoint: str) -> bool:
    """Make a DELETE request to the API."""
    try:
        with httpx.Client() as client:
            url = f"{API_V1_URL}{endpoint}"
            response = client.delete(url)

            if response.status_code == 200:
                print(f"✅ Deleted {endpoint}")
                return True
            else:
                print(f"❌ DELETE {endpoint} failed with status {response.status_code}")
                return False
    except Exception as e:
        print(f"❌ Error: {e}")
        return False


print("✅ Helper functions defined")


✅ Helper functions defined


In [4]:
# Create a sample document via API
sample_text = """Patient presents with persistent dry cough for 10 days, accompanied by fever and shortness of breath.
Chest X-ray reveals bilateral infiltrates suggestive of pneumonia.
Prescribed azithromycin 500mg daily for 5 days and albuterol inhaler for bronchodilation.
Advised bed rest and increased fluid intake."""

document_payload = {
    "title": "Case 1: Respiratory Infection",
    "text": sample_text
}

print("Creating document via API...")
doc = api_post("/documents/", document_payload)

if doc:
    doc_id = doc.get("id")
    print(f"✅ Document created: ID={doc_id}, Title='{doc.get('title')}'")
    print(f"   Text length: {len(doc.get('text', ''))} characters")
else:
    print("❌ Failed to create document")
    doc_id = None


Creating document via API...
✅ Document created: ID=3, Title='Case 1: Respiratory Infection'
   Text length: 306 characters


In [5]:
# Create annotation session via API
if doc_id:
    annotation_payload = {
        "document_id": doc_id,
        "annotator_id": "api_tester",
        "status": "in_progress"
    }

    print("\nCreating annotation via API...")
    annotation = api_post("/annotations/", annotation_payload)

    if annotation:
        ann_id = annotation.get("id")
        print(f"✅ Annotation created: ID={ann_id}")
        print(f"   Annotator: {annotation.get('annotator_id')}")
        print(f"   Status: {annotation.get('status')}")
    else:
        print("❌ Failed to create annotation")
        ann_id = None
else:
    ann_id = None



Creating annotation via API...
✅ Annotation created: ID=3
   Annotator: api_tester
   Status: in_progress


In [6]:
# Create entities via API
if ann_id:
    print("\nCreating entities via API...")

    entities_data = [
        {
            "text": "dry cough",
            "entity_type": "symptom",
            "start_char": 29,
            "end_char": 38,
            "annotation_id": ann_id,
            "confidence": 1.0
        },
        {
            "text": "fever",
            "entity_type": "symptom",
            "start_char": 70,
            "end_char": 75,
            "annotation_id": ann_id,
            "confidence": 1.0
        },
        {
            "text": "shortness of breath",
            "entity_type": "symptom",
            "start_char": 80,
            "end_char": 99,
            "annotation_id": ann_id,
            "confidence": 1.0
        },
        {
            "text": "pneumonia",
            "entity_type": "disease",
            "start_char": 155,
            "end_char": 164,
            "annotation_id": ann_id,
            "confidence": 0.95
        },
        {
            "text": "azithromycin",
            "entity_type": "medication",
            "start_char": 178,
            "end_char": 190,
            "annotation_id": ann_id,
            "confidence": 1.0
        },
        {
            "text": "500mg",
            "entity_type": "dosage",
            "start_char": 191,
            "end_char": 196,
            "annotation_id": ann_id,
            "confidence": 1.0
        },
        {
            "text": "albuterol",
            "entity_type": "medication",
            "start_char": 218,
            "end_char": 227,
            "annotation_id": ann_id,
            "confidence": 1.0
        }
    ]

    entity_ids = []
    for entity_data in entities_data:
        entity = api_post("/entities/", entity_data)
        if entity:
            entity_ids.append(entity.get("id"))
            print(f"✅ Entity created: ID={entity.get('id')}, Text='{entity.get('text')}', Type={entity.get('entity_type')}")
        else:
            print(f"❌ Failed to create entity: {entity_data['text']}")

    print(f"\n📊 Total entities created: {len(entity_ids)}")
else:
    entity_ids = []



Creating entities via API...
✅ Entity created: ID=16, Text='dry cough', Type=symptom
✅ Entity created: ID=17, Text='fever', Type=symptom
✅ Entity created: ID=18, Text='shortness of breath', Type=symptom
✅ Entity created: ID=19, Text='pneumonia', Type=disease
✅ Entity created: ID=20, Text='azithromycin', Type=medication
✅ Entity created: ID=21, Text='500mg', Type=dosage
✅ Entity created: ID=22, Text='albuterol', Type=medication

📊 Total entities created: 7


In [7]:
# Create relations via API
if ann_id and len(entity_ids) >= 7:
    print("\nCreating relations via API...")

    # Map entity IDs: 0=dry cough, 1=fever, 2=shortness of breath, 3=pneumonia, 4=azithromycin, 5=500mg, 6=albuterol

    relations_data = [
        {
            "annotation_id": ann_id,
            "source_entity_id": entity_ids[0],  # dry cough
            "target_entity_id": entity_ids[3],  # pneumonia
            "relation_type": "indicates",
            "confidence": 0.9
        },
        {
            "annotation_id": ann_id,
            "source_entity_id": entity_ids[1],  # fever
            "target_entity_id": entity_ids[3],  # pneumonia
            "relation_type": "indicates",
            "confidence": 0.95
        },
        {
            "annotation_id": ann_id,
            "source_entity_id": entity_ids[2],  # shortness of breath
            "target_entity_id": entity_ids[3],  # pneumonia
            "relation_type": "indicates",
            "confidence": 0.95
        },
        {
            "annotation_id": ann_id,
            "source_entity_id": entity_ids[4],  # azithromycin
            "target_entity_id": entity_ids[3],  # pneumonia
            "relation_type": "treats",
            "confidence": 1.0
        },
        {
            "annotation_id": ann_id,
            "source_entity_id": entity_ids[6],  # albuterol
            "target_entity_id": entity_ids[3],  # pneumonia
            "relation_type": "treats",
            "confidence": 0.85
        },
        {
            "annotation_id": ann_id,
            "source_entity_id": entity_ids[5],  # 500mg
            "target_entity_id": entity_ids[4],  # azithromycin
            "relation_type": "dosage_for",
            "confidence": 1.0
        }
    ]

    relation_ids = []
    for relation_data in relations_data:
        relation = api_post("/relations/", relation_data)
        if relation:
            relation_ids.append(relation.get("id"))
            src_id = relation.get("source_entity_id")
            tgt_id = relation.get("target_entity_id")
            rel_type = relation.get("relation_type")
            print(f"✅ Relation created: ID={relation.get('id')}, {src_id} --[{rel_type}]--> {tgt_id}")
        else:
            print("❌ Failed to create relation")

    print(f"\n📊 Total relations created: {len(relation_ids)}")
else:
    relation_ids = []



Creating relations via API...
✅ Relation created: ID=13, 16 --[indicates]--> 19
✅ Relation created: ID=14, 17 --[indicates]--> 19
✅ Relation created: ID=15, 18 --[indicates]--> 19
✅ Relation created: ID=16, 20 --[treats]--> 19
✅ Relation created: ID=17, 22 --[treats]--> 19
✅ Relation created: ID=18, 21 --[dosage_for]--> 20

📊 Total relations created: 6


## Query API Endpoints


In [8]:
# Get all documents
print("\n📄 Querying all documents...")
all_docs = api_get("/documents/")

if all_docs and isinstance(all_docs, list):
    print(f"✅ Retrieved {len(all_docs)} document(s)")

    # Convert to DataFrame for nice display
    if all_docs:
        docs_data = []
        for doc in all_docs:
            docs_data.append({
                "ID": doc.get("id"),
                "Title": doc.get("title"),
                "Text Preview": doc.get("text", "")[:50] + "..." if len(doc.get("text", "")) > 50 else doc.get("text"),
                "Text Length": len(doc.get("text", "")),
                "Created": doc.get("created_at", "")[:19]
            })

        df_docs = pd.DataFrame(docs_data)
        print("\n" + "="*80)
        display(df_docs)
else:
    print("❌ Failed to retrieve documents")



📄 Querying all documents...
✅ Retrieved 3 document(s)



,ID,Title,Text Preview,Text Length,Created
0,1,Case 1: Respiratory Infection,Patient presents with persistent dry cough for...,306,2025-11-05T21:21:36
1,2,Case 2: Diabetes Management,Patient with type 2 diabetes mellitus diagnose...,285,2025-11-05T21:21:36
2,3,Case 1: Respiratory Infection,Patient presents with persistent dry cough for...,306,2025-11-05T21:23:20


In [9]:
# Get all annotations
print("\n📝 Querying all annotations...")
all_annotations = api_get("/annotations/")

if all_annotations and isinstance(all_annotations, list):
    print(f"✅ Retrieved {len(all_annotations)} annotation(s)")

    if all_annotations:
        ann_data = []
        for ann in all_annotations:
            ann_data.append({
                "ID": ann.get("id"),
                "Document ID": ann.get("document_id"),
                "Annotator": ann.get("annotator_id"),
                "Status": ann.get("status"),
                "Updated": ann.get("updated_at", "")[:19]
            })

        df_anns = pd.DataFrame(ann_data)
        print("\n" + "="*80)
        display(df_anns)
else:
    print("❌ Failed to retrieve annotations")



📝 Querying all annotations...
✅ Retrieved 3 annotation(s)



,ID,Document ID,Annotator,Status,Updated
0,1,1,doctor_smith,in_progress,2025-11-05T21:21:36
1,2,2,doctor_johnson,in_progress,2025-11-05T21:21:37
2,3,3,api_tester,in_progress,2025-11-05T21:23:54


In [10]:
# Get all entities
print("\n🏷️  Querying all entities...")
all_entities = api_get("/entities/")

if all_entities and isinstance(all_entities, list):
    print(f"✅ Retrieved {len(all_entities)} entity(ies)")

    if all_entities:
        ent_data = []
        for ent in all_entities:
            ent_data.append({
                "ID": ent.get("id"),
                "Text": ent.get("text"),
                "Type": ent.get("entity_type"),
                "Start": ent.get("start_char"),
                "End": ent.get("end_char"),
                "Confidence": ent.get("confidence"),
                "Annotation ID": ent.get("annotation_id")
            })

        df_ents = pd.DataFrame(ent_data)
        print("\n" + "="*80)
        display(df_ents)
else:
    print("❌ Failed to retrieve entities")



🏷️  Querying all entities...
✅ Retrieved 22 entity(ies)



,ID,Text,Type,Start,End,Confidence,Annotation ID
0,1,dry cough,symptom,29,38,1.00,1
1,2,fever,symptom,70,75,1.00,1
2,3,shortness of breath,symptom,80,99,1.00,1
3,4,pneumonia,disease,155,164,0.95,1
4,5,azithromycin,medication,178,190,1.00,1
5,6,500mg,dosage,191,196,1.00,1
6,7,albuterol,medication,218,227,1.00,1
7,8,type 2 diabetes mellitus,disease,16,39,1.00,2
8,9,metformin,medication,104,113,1.00,2
9,10,1000mg,dosage,114,120,1.00,2


In [11]:
# Get all relations
print("\n🔗 Querying all relations...")
all_relations = api_get("/relations/")

if all_relations and isinstance(all_relations, list):
    print(f"✅ Retrieved {len(all_relations)} relation(s)")

    if all_relations:
        rel_data = []
        for rel in all_relations:
            rel_data.append({
                "ID": rel.get("id"),
                "Source Entity ID": rel.get("source_entity_id"),
                "Relation Type": rel.get("relation_type"),
                "Target Entity ID": rel.get("target_entity_id"),
                "Confidence": rel.get("confidence"),
                "Annotation ID": rel.get("annotation_id")
            })

        df_rels = pd.DataFrame(rel_data)
        print("\n" + "="*80)
        display(df_rels)
else:
    print("❌ Failed to retrieve relations")



🔗 Querying all relations...
✅ Retrieved 18 relation(s)



,ID,Source Entity ID,Relation Type,Target Entity ID,Confidence,Annotation ID
0,1,1,indicates,4,0.90,1
1,2,2,indicates,4,0.95,1
2,3,3,indicates,4,0.95,1
3,4,5,treats,4,1.00,1
4,5,7,treats,4,0.85,1
5,6,6,dosage_for,5,1.00,1
6,7,9,treats,8,1.00,2
7,8,11,treats,8,1.00,2
8,9,10,dosage_for,9,1.00,2
9,10,12,dosage_for,11,1.00,2


## Example 2: Create Another Document


In [12]:
# Create second document
sample_text_2 = """Patient with type 2 diabetes mellitus diagnosed 5 years ago.
Current medications include metformin 1000mg twice daily and glipizide 10mg once daily.
Recent HbA1c results show 7.8%, indicating suboptimal glycemic control.
Patient reports occasional episodes of dizziness and fatigue."""

document_payload_2 = {
    "title": "Case 2: Diabetes Management",
    "text": sample_text_2
}

print("Creating second document via API...")
doc2 = api_post("/documents/", document_payload_2)

if doc2:
    doc2_id = doc2.get("id")
    print(f"✅ Document created: ID={doc2_id}, Title='{doc2.get('title')}'")
else:
    print("❌ Failed to create document")
    doc2_id = None


Creating second document via API...
✅ Document created: ID=4, Title='Case 2: Diabetes Management'


In [13]:
# Create annotation for second document
if doc2_id:
    annotation_payload_2 = {
        "document_id": doc2_id,
        "annotator_id": "api_tester_2",
        "status": "in_progress"
    }

    print("\nCreating annotation for second document...")
    annotation2 = api_post("/annotations/", annotation_payload_2)

    if annotation2:
        ann2_id = annotation2.get("id")
        print(f"✅ Annotation created: ID={ann2_id}")
    else:
        ann2_id = None
else:
    ann2_id = None



Creating annotation for second document...
✅ Annotation created: ID=4


In [14]:
# Create entities for second document
if ann2_id:
    print("\nCreating entities for second document...")

    entities_data_2 = [
        {
            "text": "type 2 diabetes mellitus",
            "entity_type": "disease",
            "start_char": 16,
            "end_char": 39,
            "annotation_id": ann2_id,
            "confidence": 1.0
        },
        {
            "text": "metformin",
            "entity_type": "medication",
            "start_char": 104,
            "end_char": 113,
            "annotation_id": ann2_id,
            "confidence": 1.0
        },
        {
            "text": "1000mg",
            "entity_type": "dosage",
            "start_char": 114,
            "end_char": 120,
            "annotation_id": ann2_id,
            "confidence": 1.0
        },
        {
            "text": "glipizide",
            "entity_type": "medication",
            "start_char": 142,
            "end_char": 151,
            "annotation_id": ann2_id,
            "confidence": 1.0
        },
        {
            "text": "10mg",
            "entity_type": "dosage",
            "start_char": 152,
            "end_char": 156,
            "annotation_id": ann2_id,
            "confidence": 1.0
        },
        {
            "text": "HbA1c",
            "entity_type": "lab_value",
            "start_char": 185,
            "end_char": 190,
            "annotation_id": ann2_id,
            "confidence": 1.0
        },
        {
            "text": "dizziness",
            "entity_type": "symptom",
            "start_char": 230,
            "end_char": 239,
            "annotation_id": ann2_id,
            "confidence": 0.95
        },
        {
            "text": "fatigue",
            "entity_type": "symptom",
            "start_char": 244,
            "end_char": 251,
            "annotation_id": ann2_id,
            "confidence": 0.95
        }
    ]

    entity2_ids = []
    for entity_data in entities_data_2:
        entity = api_post("/entities/", entity_data)
        if entity:
            entity2_ids.append(entity.get("id"))
            print(f"✅ Entity created: ID={entity.get('id')}, Text='{entity.get('text')}'")

    print(f"\n📊 Total entities created for doc 2: {len(entity2_ids)}")
else:
    entity2_ids = []



Creating entities for second document...
✅ Entity created: ID=23, Text='type 2 diabetes mellitus'
✅ Entity created: ID=24, Text='metformin'
✅ Entity created: ID=25, Text='1000mg'
✅ Entity created: ID=26, Text='glipizide'
✅ Entity created: ID=27, Text='10mg'
✅ Entity created: ID=28, Text='HbA1c'
✅ Entity created: ID=29, Text='dizziness'
✅ Entity created: ID=30, Text='fatigue'

📊 Total entities created for doc 2: 8


In [15]:
# Create relations for second document
if ann2_id and len(entity2_ids) >= 8:
    print("\nCreating relations for second document...")

    # Map: 0=type2diabetes, 1=metformin, 2=1000mg, 3=glipizide, 4=10mg, 5=HbA1c, 6=dizziness, 7=fatigue

    relations_data_2 = [
        {
            "annotation_id": ann2_id,
            "source_entity_id": entity2_ids[1],  # metformin
            "target_entity_id": entity2_ids[0],  # type 2 diabetes
            "relation_type": "treats",
            "confidence": 1.0
        },
        {
            "annotation_id": ann2_id,
            "source_entity_id": entity2_ids[3],  # glipizide
            "target_entity_id": entity2_ids[0],  # type 2 diabetes
            "relation_type": "treats",
            "confidence": 1.0
        },
        {
            "annotation_id": ann2_id,
            "source_entity_id": entity2_ids[2],  # 1000mg
            "target_entity_id": entity2_ids[1],  # metformin
            "relation_type": "dosage_for",
            "confidence": 1.0
        },
        {
            "annotation_id": ann2_id,
            "source_entity_id": entity2_ids[4],  # 10mg
            "target_entity_id": entity2_ids[3],  # glipizide
            "relation_type": "dosage_for",
            "confidence": 1.0
        },
        {
            "annotation_id": ann2_id,
            "source_entity_id": entity2_ids[0],  # type 2 diabetes
            "target_entity_id": entity2_ids[6],  # dizziness
            "relation_type": "has_symptom",
            "confidence": 0.8
        },
        {
            "annotation_id": ann2_id,
            "source_entity_id": entity2_ids[0],  # type 2 diabetes
            "target_entity_id": entity2_ids[7],  # fatigue
            "relation_type": "has_symptom",
            "confidence": 0.85
        }
    ]

    for relation_data in relations_data_2:
        relation = api_post("/relations/", relation_data)
        if relation:
            print(f"✅ Relation created: ID={relation.get('id')}")

    print("\n✅ All relations created for doc 2")



Creating relations for second document...
✅ Relation created: ID=19
✅ Relation created: ID=20
✅ Relation created: ID=21
✅ Relation created: ID=22
✅ Relation created: ID=23
✅ Relation created: ID=24

✅ All relations created for doc 2


## Database Summary


In [16]:
def print_api_summary():
    """Print a summary of the API database contents."""
    docs = api_get("/documents/") or []
    anns = api_get("/annotations/") or []
    ents = api_get("/entities/") or []
    rels = api_get("/relations/") or []

    print("\n" + "="*50)
    print("📊 DATABASE SUMMARY (via API)")
    print("="*50)
    print(f"📄 Documents:   {len(docs) if isinstance(docs, list) else 0}")
    print(f"📝 Annotations: {len(anns) if isinstance(anns, list) else 0}")
    print(f"🏷️  Entities:    {len(ents) if isinstance(ents, list) else 0}")
    print(f"🔗 Relations:   {len(rels) if isinstance(rels, list) else 0}")
    print("="*50 + "\n")

print_api_summary()



📊 DATABASE SUMMARY (via API)
📄 Documents:   4
📝 Annotations: 4
🏷️  Entities:    30
🔗 Relations:   24



## View All Data from API


In [17]:
print("\n📄 ALL DOCUMENTS (from API):")
print("="*80)
all_docs_final = api_get("/documents/") or []
if all_docs_final and isinstance(all_docs_final, list):
    docs_final_data = []
    for doc in all_docs_final:
        docs_final_data.append({
            "ID": doc.get("id"),
            "Title": doc.get("title"),
            "Text": doc.get("text", "")[:60] + "..." if len(doc.get("text", "")) > 60 else doc.get("text"),
        })
    df = pd.DataFrame(docs_final_data)
    display(df)



📄 ALL DOCUMENTS (from API):


,ID,Title,Text
0,1,Case 1: Respiratory Infection,Patient presents with persistent dry cough for...
1,2,Case 2: Diabetes Management,Patient with type 2 diabetes mellitus diagnose...
2,3,Case 1: Respiratory Infection,Patient presents with persistent dry cough for...
3,4,Case 2: Diabetes Management,Patient with type 2 diabetes mellitus diagnose...


In [18]:
print("\n📝 ALL ANNOTATIONS (from API):")
print("="*80)
all_anns_final = api_get("/annotations/") or []
if all_anns_final and isinstance(all_anns_final, list):
    anns_final_data = []
    for ann in all_anns_final:
        anns_final_data.append({
            "ID": ann.get("id"),
            "Document ID": ann.get("document_id"),
            "Annotator": ann.get("annotator_id"),
            "Status": ann.get("status"),
        })
    df = pd.DataFrame(anns_final_data)
    display(df)



📝 ALL ANNOTATIONS (from API):


,ID,Document ID,Annotator,Status
0,1,1,doctor_smith,in_progress
1,2,2,doctor_johnson,in_progress
2,3,3,api_tester,in_progress
3,4,4,api_tester_2,in_progress


In [19]:
print("\n🏷️  ALL ENTITIES (from API):")
print("="*80)
all_ents_final = api_get("/entities/") or []
if all_ents_final and isinstance(all_ents_final, list):
    ents_final_data = []
    for ent in all_ents_final:
        ents_final_data.append({
            "ID": ent.get("id"),
            "Text": ent.get("text"),
            "Type": ent.get("entity_type"),
            "Start": ent.get("start_char"),
            "End": ent.get("end_char"),
            "Confidence": ent.get("confidence"),
        })
    df = pd.DataFrame(ents_final_data)
    display(df)



🏷️  ALL ENTITIES (from API):


,ID,Text,Type,Start,End,Confidence
0,1,dry cough,symptom,29,38,1.00
1,2,fever,symptom,70,75,1.00
2,3,shortness of breath,symptom,80,99,1.00
3,4,pneumonia,disease,155,164,0.95
4,5,azithromycin,medication,178,190,1.00
5,6,500mg,dosage,191,196,1.00
6,7,albuterol,medication,218,227,1.00
7,8,type 2 diabetes mellitus,disease,16,39,1.00
8,9,metformin,medication,104,113,1.00
9,10,1000mg,dosage,114,120,1.00


In [20]:
print("\n🔗 ALL RELATIONS (from API):")
print("="*80)
all_rels_final = api_get("/relations/") or []
if all_rels_final and isinstance(all_rels_final, list):
    rels_final_data = []
    for rel in all_rels_final:
        rels_final_data.append({
            "ID": rel.get("id"),
            "Source Entity ID": rel.get("source_entity_id"),
            "Relation Type": rel.get("relation_type"),
            "Target Entity ID": rel.get("target_entity_id"),
            "Confidence": rel.get("confidence"),
        })
    df = pd.DataFrame(rels_final_data)
    display(df)



🔗 ALL RELATIONS (from API):


,ID,Source Entity ID,Relation Type,Target Entity ID,Confidence
0,1,1,indicates,4,0.90
1,2,2,indicates,4,0.95
2,3,3,indicates,4,0.95
3,4,5,treats,4,1.00
4,5,7,treats,4,0.85
5,6,6,dosage_for,5,1.00
6,7,9,treats,8,1.00
7,8,11,treats,8,1.00
8,9,10,dosage_for,9,1.00
9,10,12,dosage_for,11,1.00


## Test Individual Endpoints


In [21]:
# Test getting specific document
if doc_id:
    print(f"\n🔍 Getting document {doc_id} from API...")
    doc_detail = api_get(f"/documents/{doc_id}")

    if doc_detail:
        print("✅ Retrieved document:")
        print(f"   ID: {doc_detail.get('id')}")
        print(f"   Title: {doc_detail.get('title')}")
        print(f"   Text length: {len(doc_detail.get('text', ''))}")
        print(f"   Created: {doc_detail.get('created_at')}")
else:
    print("No document ID available")



🔍 Getting document 3 from API...
✅ Retrieved document:
   ID: 3
   Title: Case 1: Respiratory Infection
   Text length: 306
   Created: 2025-11-05T21:23:20.972807


In [22]:
# Test getting specific annotation
if ann_id:
    print(f"\n🔍 Getting annotation {ann_id} from API...")
    ann_detail = api_get(f"/annotations/{ann_id}")

    if ann_detail:
        print("✅ Retrieved annotation:")
        print(f"   ID: {ann_detail.get('id')}")
        print(f"   Document ID: {ann_detail.get('document_id')}")
        print(f"   Annotator: {ann_detail.get('annotator_id')}")
        print(f"   Status: {ann_detail.get('status')}")
        print(f"   Updated: {ann_detail.get('updated_at')}")
else:
    print("No annotation ID available")



🔍 Getting annotation 3 from API...
✅ Retrieved annotation:
   ID: 3
   Document ID: 3
   Annotator: api_tester
   Status: in_progress
   Updated: 2025-11-05T21:23:54.067680


In [23]:
# Test filtering entities by annotation
if ann_id:
    print(f"\n🔍 Getting entities for annotation {ann_id}...")
    ann_entities = api_get("/entities/", {"annotation_id": ann_id})

    if ann_entities and isinstance(ann_entities, list):
        print(f"✅ Retrieved {len(ann_entities)} entities for this annotation")
        for ent in ann_entities[:3]:  # Show first 3
            print(f"   - {ent.get('text')} ({ent.get('entity_type')})")
        if len(ann_entities) > 3:
            print(f"   ... and {len(ann_entities) - 3} more")
else:
    print("No annotation ID available")



🔍 Getting entities for annotation 3...
✅ Retrieved 7 entities for this annotation
   - dry cough (symptom)
   - fever (symptom)
   - shortness of breath (symptom)
   ... and 4 more


## API Response Examples (JSON)


In [24]:
# Show raw JSON response
if doc_id:
    print(f"\n📋 Raw API Response for Document {doc_id}:")
    print("="*80)
    doc_response = api_get(f"/documents/{doc_id}")
    if doc_response:
        print(json.dumps(doc_response, indent=2, default=str))
else:
    print("No document to show")



📋 Raw API Response for Document 3:
{
  "title": "Case 1: Respiratory Infection",
  "text": "Patient presents with persistent dry cough for 10 days, accompanied by fever and shortness of breath. \nChest X-ray reveals bilateral infiltrates suggestive of pneumonia. \nPrescribed azithromycin 500mg daily for 5 days and albuterol inhaler for bronchodilation. \nAdvised bed rest and increased fluid intake.",
  "updated_at": "2025-11-05T21:23:20.972821",
  "created_at": "2025-11-05T21:23:20.972807",
  "id": 3
}


In [ ]:
# Show raw JSON response for annotation
if ann_id:
    print(f"\n📋 Raw API Response for Annotation {ann_id}:")
    print("="*80)
    ann_response = api_get(f"/annotations/{ann_id}")
    if ann_response:
        print(json.dumps(ann_response, indent=2, default=str))
else:
    print("No annotation to show")


In [ ]:
# Show raw JSON response for first entity
if len(entity_ids) > 0:
    print(f"\n📋 Raw API Response for First Entity (ID {entity_ids[0]}):")
    print("="*80)
    ent_response = api_get(f"/entities/{entity_ids[0]}")
    if ent_response:
        print(json.dumps(ent_response, indent=2, default=str))
else:
    print("No entities to show")


## Template: Using API Programmatically

Here's how to use the API endpoints in your own code.


In [ ]:
# TEMPLATE: How to use the API programmatically

template_code = '''
import httpx
import json

# API Configuration
API_BASE_URL = "http://localhost:8000"
API_V1_URL = f"{API_BASE_URL}/api/v1"

# Example 1: Create a document
with httpx.Client() as client:
    doc_data = {
        "title": "My Medical Case",
        "text": "Patient presents with..."
    }
    response = client.post(f"{API_V1_URL}/documents/", json=doc_data)
    document = response.json()
    doc_id = document["id"]

# Example 2: Create annotation for the document
with httpx.Client() as client:
    ann_data = {
        "document_id": doc_id,
        "annotator_id": "my_name",
        "status": "in_progress"
    }
    response = client.post(f"{API_V1_URL}/annotations/", json=ann_data)
    annotation = response.json()
    ann_id = annotation["id"]

# Example 3: Add an entity
with httpx.Client() as client:
    entity_data = {
        "text": "disease_name",
        "entity_type": "disease",
        "start_char": 0,
        "end_char": 12,
        "annotation_id": ann_id,
        "confidence": 1.0
    }
    response = client.post(f"{API_V1_URL}/entities/", json=entity_data)
    entity = response.json()

# Example 4: Create a relation between entities
with httpx.Client() as client:
    relation_data = {
        "annotation_id": ann_id,
        "source_entity_id": entity_id_1,
        "target_entity_id": entity_id_2,
        "relation_type": "treats",
        "confidence": 0.95
    }
    response = client.post(f"{API_V1_URL}/relations/", json=relation_data)
    relation = response.json()

# Example 5: Query all documents
with httpx.Client() as client:
    response = client.get(f"{API_V1_URL}/documents/")
    documents = response.json()

# Example 6: Get specific document
with httpx.Client() as client:
    response = client.get(f"{API_V1_URL}/documents/{doc_id}")
    document = response.json()

# Example 7: Update document
with httpx.Client() as client:
    update_data = {"title": "New Title"}
    response = client.put(f"{API_V1_URL}/documents/{doc_id}", json=update_data)
    updated_doc = response.json()

# Example 8: Delete a document
with httpx.Client() as client:
    response = client.delete(f"{API_V1_URL}/documents/{doc_id}")
    # Returns {"message": "Document deleted successfully"}
'''

print("📝 TEMPLATE CODE:")
print("="*80)
print(template_code)


In [ ]:
def check_wipe_db():
    """Check if wiping DB is working."""
    try:
        with httpx.Client() as client:
            response = client.post(f"{API_BASE_URL}/wipe_db")
            if response.status_code == 200:
                print("✅ DB Wiped!")
                data = response.json()
                print(f"   Status: {data.get('status')}")
                return True
            else:
                print(f"❌ API returned status {response.status_code}")
                return False
    except Exception as e:
        print(f"❌ Cannot connect to API: {e}")
        print(f"   Make sure the API is running at {API_BASE_URL}")
        print("   Run: python run.py")
        return False

# Check API health
wipe_ok = check_wipe_db()